Flow: **Setup → Probe → Run (foreground) → Validate → Download**.


In [ ]:
# 1) SETUP — run once per machine
import os
import sys

BRANCH = "vibe/cma-robust-speed-911692"   # training code + new genome
REPO_URL = "https://github.com/merwynnn/AutonomousCar.git"

if not os.path.exists("training/cma_opt.py") and not os.path.isdir("AutonomousCar"):
    !git clone -b {BRANCH} {REPO_URL}
    %cd AutonomousCar
elif not os.path.exists("training/cma_opt.py"):
    %cd AutonomousCar
!git pull -q
# One BLAS thread per rollout worker: prevents thread thrash across workers.
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[var] = "1"
!{sys.executable} -m pip install -q -e ".[train,dev]"
print("cwd:", os.getcwd(), "| cores available:", os.cpu_count())

In [ ]:
# 2) QUICK BENCHMARK — cost of one rollout on this machine (~1-2 s/core is normal)
import time

from training.env import RolloutConfig, run_vector
from training.export import load_policy

_, v, _, _ = load_policy("algorithms/weights/policy_robust_b1.npz")
t0 = time.perf_counter()
r = run_vector(v, RolloutConfig(policy_type="b1", subrays=1, seed=42, max_time_s=20))
dt = time.perf_counter() - t0
print(f"one lap rollout: {dt:.2f} s wall | lap {r.lap_time_s} s | contacts {r.contacts}")
print(f"estimated throughput: ~{os.cpu_count()/dt:.0f} rollouts/s with {os.cpu_count()} workers")

## 3) RUN
The training runs **in the foreground of this cell**: live progress prints below it, the cell stays busy (which keeps hosted sessions like Kaggle alive), and an interrupt stops it cleanly with all results kept. No background processes.


## 2b) Hardware probe
Run this before launching: it computes a **safe worker count** from the CPU affinity, the container CPU quota and the available RAM. The earlier crash was almost certainly the parent being OOM-killed after 24 MuJoCo workers (one per *physical* core) exceeded the slice's memory/CPU quota.


In [ ]:
# 2b) HARDWARE PROBE — how many workers can this slice really sustain?
import os

# CPU affinity (respects cpusets) and cgroup v2 quota
try:
    affinity = len(os.sched_getaffinity(0))
except AttributeError:
    affinity = os.cpu_count()
quota_cores = None
try:
    with open("/sys/fs/cgroup/cpu.max") as f:
        q, per = f.read().split()[:2]
    if q != "max":
        quota_cores = max(1, int(q) // int(per))
except OSError:
    pass

# Usable RAM (cgroup limit or total)
mem_gb = None
try:
    with open("/sys/fs/cgroup/memory.max") as f:
        mem_gb = int(f.read()) / 1e9
except (OSError, ValueError):
    pass
if mem_gb is None:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                mem_gb = int(line.split()[1]) / 1e6

cpu_workers = min(affinity, quota_cores or affinity)
# Each rollout worker holds a MuJoCo world: budget ~0.6 GB per worker and
# leave headroom for the notebook kernel.
mem_workers = max(1, int((mem_gb - 2.0) / 0.6))
WORKERS = max(1, min(cpu_workers, mem_workers))
print(f"affinity: {affinity} | cgroup quota: {quota_cores} | RAM: {mem_gb:.1f} GB")
print(f"safe worker count: {WORKERS} (cpu cap {cpu_workers}, memory cap {mem_workers})")


In [ ]:
# 3) RUN TRAINING (foreground) — the cell stays busy for the whole run
# Progress prints live below; interrupt the cell to stop early (results so
# far are already checkpointed in data/training/).
import time

from training import cma_opt

GENS = 600          # generation cap
BUDGET_H = 6.0      # wall-clock cap in hours
STALL = 90          # early stop: generations without a promotion
POPSIZE = 24
K = 6               # tracks per generation (reference + K-1 random)
# WORKERS comes from the HARDWARE PROBE cell; auto-detect as fallback.
try:
    WORKERS = WORKERS  # noqa: B018 - probe ran? keep it
except NameError:
    WORKERS = 0        # 0 = auto-detect (cgroup/affinity aware)

argv = ["cma_opt",
    "--stage", "b1", "--generations", str(GENS), "--popsize", str(POPSIZE),
    "--workers", str(WORKERS),
    "--random-track", "--tracks-per-generation", str(K),
    "--fitness-aggregation", "worst", "--vary-sensor-seed",
    "--reference-track-prob", "0.15",
    "--object-density-range", "0.0", "0.95",
    "--object-scale-range", "1.0", "1.9",
    "--track-width-range", "0.82", "1.0",
    "--init", "algorithms/weights/policy_robust_b1.npz",
    "--sigma0", "0.20", "--seed", "42", "--max-time", "20",
    "--out", "data/training/b1_remote_best.npz",
    "--budget-seconds", str(int(BUDGET_H * 3600)),
    "--stop-after-stall", str(STALL),
]
old_argv = sys.argv
sys.argv = argv
try:
    cma_opt.main()          # runs the search; prints live progress below
except SystemExit as exit_code:
    print(f"training finished with exit code {exit_code.code}")
finally:
    sys.argv = old_argv


In [ ]:
# 4) MONITOR — run this cell any time (safe to repeat)
import json
import time

try:
    with open("data/training/b1_status.json") as f:
        st = json.load(f)
    print(f"gen {st.get('generation')}/{st.get('generations')} | "
          f"best lap {st.get('best_lap_s')} s | contacts {st.get('best_contacts')} | "
          f"elapsed {time.strftime('%H:%M:%S', time.gmtime(st.get('elapsed_s', 0)))}")
except FileNotFoundError:
    print("no status yet")
print("--- last lines ---")
with open("data/training/remote.log") as f:
    print("".join(f.readlines()[-4:]))


In [ ]:
# 5) STOP — interrupt the RUN cell (Kernel > Interrupt) to stop early.
# Results so far are kept: the best vector is checkpointed every generation
# in data/training/b1_checkpoint.pkl and the objective winner in
# data/training/b1_objective_best.npz.
import subprocess

subprocess.run(["pkill", "-f", "training.cma_opt"], check=False)
print("any leftover background trainers killed; interrupt the RUN cell if it is still busy")


## 6) VALIDATE

Held-out checks on the winner: reference lap across sensor-noise seeds, then
fresh random tracks (density 0–0.9, object scale 1.0–1.8 — the hard
distribution the robustness objective trains against).


In [ ]:
# 6) VALIDATE — run after (or during) training
BEST = "data/training/b1_objective_best.npz"   # robustness-first winner
!{sys.executable} -m training.evaluate --policy {BEST} --subrays 1 --max-time 30 --seed-sweep 12
!{sys.executable} -m training.evaluate --policy {BEST} --subrays 1 3 --max-time 40
!{sys.executable} -m training.evaluate --policy {BEST} --subrays 1 --max-time 40 --random-sweep 20 --track-seed 999

In [ ]:
# 7) DOWNLOAD — grab these files when the run finishes
for f in ["data/training/b1_objective_best.npz",   # robustness-first winner
          "data/training/b1_remote_best.npz",       # best clean reference lap
          "data/training/b1_log.jsonl",             # full generation history
          "data/training/b1_checkpoint.pkl"]:
    print(f, os.path.exists(f) and os.path.getsize(f), "bytes")
print("\nUpload the .npz files back (GitHub PR or shared drive);")
print("data/training/b1_objective_best.npz is the candidate to ship.")